In [3]:
from google.colab import files
uploaded=files.upload()

Saving Multilingual_Hate_Speech_Dataset.xlsx to Multilingual_Hate_Speech_Dataset.xlsx


In [4]:
import pandas as pd

df_en = pd.read_excel("Multilingual_Hate_Speech_Dataset.xlsx", sheet_name="English")
df_hi = pd.read_excel("Multilingual_Hate_Speech_Dataset.xlsx", sheet_name="Hindi")
df_mr = pd.read_excel("Multilingual_Hate_Speech_Dataset.xlsx", sheet_name="Marathi")

df = pd.concat([df_en, df_hi, df_mr], ignore_index=True)

In [5]:
# Create binary class (1 = toxic, 0 = normal)
df['class'] = ((df['Hate_Speech'] > 0) | (df['Offensive_language'] > 0)).astype(int)

# Remove missing labels
df = df.dropna(subset=['class'])

# Keep required columns
df = df[['Text', 'class']]

# Shuffle dataset (VERY IMPORTANT)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Clean text
X = df['Text'].fillna("").astype(str)
y = df['class']

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [7]:
# Count Vectorizer / TF-IDF setup

X = df['Text'].fillna("").astype(str)   # ✅ FIX
y = df['class']

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_vec = vectorizer.fit_transform(X)

In [8]:
#TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [9]:
#Naive-Byes(Count Vectorizer)
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(max_features=5000, ngram_range=(1,2))

X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb_cv = MultinomialNB()
nb_cv.fit(X_train_cv, y_train)

y_pred_nb_cv = nb_cv.predict(X_test_cv)

print("=== Naive Bayes (CountVectorizer) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb_cv))
print(classification_report(y_test, y_pred_nb_cv))
print(confusion_matrix(y_test, y_pred_nb_cv))

=== Naive Bayes (CountVectorizer) ===
Accuracy: 0.6440545808966862
              precision    recall  f1-score   support

           0       0.66      0.53      0.59      1219
           1       0.64      0.75      0.69      1346

    accuracy                           0.64      2565
   macro avg       0.65      0.64      0.64      2565
weighted avg       0.65      0.64      0.64      2565

[[ 645  574]
 [ 339 1007]]


In [10]:
#Naive Bayes (TF-IDF)
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

print("=== Naive Bayes (TF-IDF) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))
print(confusion_matrix(y_test, y_pred_nb))

=== Naive Bayes (TF-IDF) ===
Accuracy: 0.6491228070175439
              precision    recall  f1-score   support

           0       0.67      0.51      0.58      1219
           1       0.64      0.77      0.70      1346

    accuracy                           0.65      2565
   macro avg       0.65      0.64      0.64      2565
weighted avg       0.65      0.65      0.64      2565

[[ 625  594]
 [ 306 1040]]


In [11]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)

print("=== Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))

=== Logistic Regression ===
Accuracy: 0.6791423001949318
              precision    recall  f1-score   support

           0       0.68      0.61      0.64      1219
           1       0.68      0.74      0.71      1346

    accuracy                           0.68      2565
   macro avg       0.68      0.68      0.68      2565
weighted avg       0.68      0.68      0.68      2565

[[743 476]
 [347 999]]


In [12]:
#SVM
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("=== Linear SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))
print(confusion_matrix(y_test, y_pred_svm))

=== Linear SVM ===
Accuracy: 0.6974658869395711
              precision    recall  f1-score   support

           0       0.69      0.65      0.67      1219
           1       0.70      0.74      0.72      1346

    accuracy                           0.70      2565
   macro avg       0.70      0.70      0.70      2565
weighted avg       0.70      0.70      0.70      2565

[[797 422]
 [354 992]]


In [13]:
#XG Boost
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Fix labels
y_train = y_train.astype(int)
y_test = y_test.astype(int)

# Convert sparse → dense
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

# Train
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_dense, y_train)

# Predict
y_pred_xgb = xgb.predict(X_test_dense)

# Evaluate
print("=== XGBoost ===")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [03:28:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
Accuracy: 0.6760233918128655
              precision    recall  f1-score   support

           0       0.66      0.65      0.66      1219
           1       0.69      0.70      0.69      1346

    accuracy                           0.68      2565
   macro avg       0.68      0.68      0.68      2565
weighted avg       0.68      0.68      0.68      2565

[[798 421]
 [410 936]]


In [14]:
#compare models
print("=== Model Comparison ===")
print("NB (CountVectorizer):", accuracy_score(y_test, y_pred_nb_cv))
print("NB (TF-IDF):", accuracy_score(y_test, y_pred_nb))
print("Logistic Regression:", accuracy_score(y_test, y_pred_lr))
print("Linear SVM:", accuracy_score(y_test, y_pred_svm))
print("XGBoost:", accuracy_score(y_test, y_pred_xgb))


=== Model Comparison ===
NB (CountVectorizer): 0.6440545808966862
NB (TF-IDF): 0.6491228070175439
Logistic Regression: 0.6791423001949318
Linear SVM: 0.6974658869395711
XGBoost: 0.6760233918128655


In [15]:
#Grid Search on SVM
from sklearn.model_selection import GridSearchCV

params = {'C': [0.1, 1, 10]}

grid = GridSearchCV(LinearSVC(), params, cv=3)
grid.fit(X_train_tfidf, y_train)

best_svm = grid.best_estimator_

print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Params: {'C': 1}
Best Score: 0.6863303560747421


In [16]:
y_pred_best = best_svm.predict(X_test_tfidf)

print("=== Tuned SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))
print(confusion_matrix(y_test, y_pred_best))


=== Tuned SVM ===
Accuracy: 0.6974658869395711
              precision    recall  f1-score   support

           0       0.69      0.65      0.67      1219
           1       0.70      0.74      0.72      1346

    accuracy                           0.70      2565
   macro avg       0.70      0.70      0.70      2565
weighted avg       0.70      0.70      0.70      2565

[[797 422]
 [354 992]]


In [17]:
#pipeline
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('model', LinearSVC())
])

pipeline.fit(X_train, y_train)

y_pred_pipe = pipeline.predict(X_test)

print("=== Pipeline SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_pipe))
print(classification_report(y_test, y_pred_pipe))
print(confusion_matrix(y_test, y_pred_pipe))

=== Pipeline SVM ===
Accuracy: 0.6974658869395711
              precision    recall  f1-score   support

           0       0.69      0.65      0.67      1219
           1       0.70      0.74      0.72      1346

    accuracy                           0.70      2565
   macro avg       0.70      0.70      0.70      2565
weighted avg       0.70      0.70      0.70      2565

[[797 422]
 [354 992]]


In [18]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip


--2026-06-27 03:29:11--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-06-27 03:29:11--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-06-27 03:29:12--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [19]:
import numpy as np

embeddings = {}

with open("glove.6B.50d.txt", encoding='utf-8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.array(values[1:], dtype='float32')
        embeddings[word] = vector

print("Loaded words:", len(embeddings))

Loaded words: 400000


In [20]:
def text_to_vector(text):
    words = text.split()
    vectors = []

    for word in words:
        if word in embeddings:
            vectors.append(embeddings[word])

    if len(vectors) == 0:
        return np.zeros(50)   # 50d vector

    return np.mean(vectors, axis=0)

In [21]:
# IMPORTANT: clean text again
X = df['Text'].fillna("").astype(str)
y = df['class'].astype(int)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
X_train_glove = np.array([text_to_vector(text) for text in X_train])
X_test_glove = np.array([text_to_vector(text) for text in X_test])

In [23]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train SVM on GloVe features
svm_glove = LinearSVC()
svm_glove.fit(X_train_glove, y_train)

# Predict
y_pred_svm_glove = svm_glove.predict(X_test_glove)

# Evaluate
print("=== GloVe + SVM ===")
print("Accuracy:", accuracy_score(y_test, y_pred_svm_glove))
print(classification_report(y_test, y_pred_svm_glove))
print(confusion_matrix(y_test, y_pred_svm_glove))

=== GloVe + SVM ===
Accuracy: 0.5875243664717349
              precision    recall  f1-score   support

           0       0.62      0.34      0.44      1219
           1       0.58      0.81      0.67      1346

    accuracy                           0.59      2565
   macro avg       0.60      0.58      0.56      2565
weighted avg       0.60      0.59      0.56      2565

[[ 413  806]
 [ 252 1094]]


In [24]:
# =========================
# MANUAL INPUT
# =========================
text = ["I hate USA"]

# =========================
# COUNT VECTORIZER
# =========================
text_cv = cv.transform(text)
pred_nb_cv = nb_cv.predict(text_cv)

# =========================
# TF-IDF
# =========================
text_tfidf = tfidf.transform(text)

pred_nb = nb.predict(text_tfidf)
pred_lr = lr.predict(text_tfidf)
pred_svm = svm.predict(text_tfidf)
pred_xgb = xgb.predict(text_tfidf.toarray())   # XGB needs dense

# =========================
# PIPELINE
# =========================
pred_pipe = pipeline.predict(text)

# =========================
# GLOVE (SVM)
# =========================
import numpy as np

text_glove = np.array([text_to_vector(text[0])])
pred_glove_svm = svm_glove.predict(text_glove)

# =========================
# LABEL FUNCTION
# =========================
def label(x):
    return "Toxic" if x == 1 else "Normal"

# =========================
# PRINT RESULTS
# =========================
print("Input:", text[0])
print("\n--- Predictions ---\n")

print("Naive Bayes (CountVectorizer):", label(pred_nb_cv[0]))
print("Naive Bayes (TF-IDF):", label(pred_nb[0]))
print("Logistic Regression:", label(pred_lr[0]))
print("Linear SVM:", label(pred_svm[0]))
print("XGBoost:", label(pred_xgb[0]))
print("Pipeline SVM:", label(pred_pipe[0]))
print("GloVe + SVM:", label(pred_glove_svm[0]))

Input: I hate USA

--- Predictions ---

Naive Bayes (CountVectorizer): Normal
Naive Bayes (TF-IDF): Normal
Logistic Regression: Toxic
Linear SVM: Normal
XGBoost: Normal
Pipeline SVM: Normal
GloVe + SVM: Toxic
